# Lesson 15a: Efficient and Scalable Deep Learning — Theory

Every notebook so far has assumed a model and its data simply fit: in memory,
on one device, inside a training budget nobody had to think about. Real
training runs run out of one or more of these long before they run out of
ideas — a model that fits on paper does not fit in a GPU's memory once
gradients and optimiser state are added on top of the parameters, a forward
pass computed in `fp32` costs twice the memory and (on hardware built for it)
half the throughput an equivalent `fp16` pass would, and a model with enough
parameters simply does not fit on any single device at all. This notebook
derives the arithmetic behind four of the standard responses — mixed-precision
training, quantisation, distillation, and splitting the work itself across
devices — and grounds each one in the same tiny CIFAR-10 network, so every
number below is computed, not asserted.

By the end of this notebook you will have:
- accounted for a training step's memory exactly, in **parameters, gradients,
  optimiser state, and (measured with forward hooks, not just claimed)
  activations** — and confirmed the last of these, the only one that depends
  on batch size, scales linearly,
- derived why **`fp16` gradients silently underflow to zero**, confirmed loss
  scaling recovers them, and shown why some computations (accumulation) must
  stay in a wider format regardless of loss scaling,
- derived **int8 quantisation's scale and zero-point** from first principles,
  and shown exactly how a single outlier value in a tensor wastes precision on
  every other, typical value in the same tensor,
- trained a teacher and a much smaller student, with and without
  **distillation**, and verified what the distillation loss actually
  optimises for (agreement with the teacher's output distribution) directly,
  rather than assuming it shows up as a test-accuracy win in a toy setup, and
- computed, from this notebook's own memory arithmetic, the GPU count a
  hypothetical large model would force even before training starts —
  motivating **model parallelism** as a memory constraint that **data
  parallelism** cannot solve, rather than merely a speed optimisation.

## Introduction

Every architecture this series has built — MLPs, convnets, residual networks,
transformers — has so far been discussed as if compute and memory were free.
They are not, and the constraints they impose are not incidental: they decide
whether a model can train at all on the hardware available, how large a batch
fits, and how many devices a single run actually needs. Four techniques
answer different parts of this. **Mixed precision** trades numerical range
for speed and memory by computing (most of) a forward and backward pass in a
narrower floating-point format. **Quantisation** goes further, representing
weights (and sometimes activations) in as little as 8 bits for storage and
inference. **Distillation** shrinks a model outright, training a small
"student" to match a larger "teacher"'s output distribution rather than
labels alone. **Parallelism** stops trying to fit everything on one device and
instead splits the *data* (every device holds the whole model, on a different
batch shard) or the *model itself* (every device holds only part of the
model) across several devices. Each section below derives one of these
concretely enough to compute with, not just describe.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, minibatch sampling,
# data subsampling) is reproducible.
import io
import pathlib
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

### Loading a CIFAR-10 Subset

A small CIFAR-10 subset — 2,000 training images, 500 held out for evaluation
— gives the teacher/student experiment in the Distillation section a real (if
tiny) classification problem, and gives every earlier section's memory
arithmetic a concrete `(3, 32, 32)` input to compute with. Loaded via the same
Hugging Face parquet mirror 5a/5b used for the unreliably slow canonical
torchvision CIFAR-10 host.

In [ ]:
CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float32) / 255.0
        for i in idx
    ])
    labels = df.iloc[idx]["label"].to_numpy()
    return images.transpose(0, 3, 1, 2), labels  # (N, C, H, W)


X_train, y_train = load_cifar10_subset("train", 2000, seed=SEED)
X_test, y_test = load_cifar10_subset("test", 500, seed=SEED)
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test, dtype=torch.long)
print(f"train: {X_train_t.shape}, test: {X_test_t.shape}, classes: {len(set(y_train.tolist()))}")

## Compute and Memory Accounting

A single training step holds four kinds of tensor in memory simultaneously,
and only one of them depends on how many examples are in a batch:

- **Parameters** ($P$ scalars): the weights themselves, one copy.
- **Gradients**: exactly one gradient per parameter — same shape, same count,
  so the same number of bytes as the parameters.
- **Optimiser state**: SGD with momentum needs one extra buffer per parameter
  (1×); Adam needs two, a running mean and a running variance of the gradient
  (2×). This state exists purely to make the optimiser's update rule work — it
  costs memory whether or not it changes what the model computes.
- **Activations**: every intermediate layer output kept from the forward
  pass, because the backward pass needs it to evaluate the chain rule locally
  at that layer. Unlike the first three, this scales with **batch size** —
  twice the images in a batch means twice the stored activations, but exactly
  the same parameters, gradients and optimiser state.

For a model with $P$ parameters stored in 4-byte `fp32` and trained with
Adam, the batch-size-independent memory is
$$\text{bytes}_{\text{fixed}} = \underbrace{4P}_{\text{params}} + \underbrace{4P}_{\text{grads}} + \underbrace{2\times 4P}_{\text{Adam state}} = 16P \text{ bytes.}$$

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(32 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        return self.fc(x.flatten(1))


model = TinyCNN()
n_params = sum(p.numel() for p in model.parameters())
bytes_per_param = 4  # fp32
param_bytes = n_params * bytes_per_param
grad_bytes = param_bytes
adam_state_bytes = 2 * param_bytes
fixed_bytes = param_bytes + grad_bytes + adam_state_bytes

print(f"TinyCNN: {n_params:,} parameters")
print(f"  params={param_bytes:,} B  grads={grad_bytes:,} B  Adam state={adam_state_bytes:,} B")
print(f"  fixed (batch-independent) total = {fixed_bytes:,} B ({fixed_bytes/1e6:.3f} MB) = 16 x {n_params:,}")
assert fixed_bytes == 16 * n_params

Measuring, rather than assuming, that activation memory scales linearly with
batch size: a forward hook on every leaf module records the byte size of its
output tensor, summed across the whole network for a given batch.

In [ ]:
activation_bytes = {}


def make_hook(name):
    def hook(module, inp, out):
        activation_bytes[name] = out.numel() * out.element_size()
    return hook


for name, module in model.named_modules():
    if len(list(module.children())) == 0:
        module.register_forward_hook(make_hook(name))

batch_sizes = [1, 8, 32, 128]
activation_totals = []
for batch in batch_sizes:
    activation_bytes.clear()
    with torch.no_grad():
        model(torch.randn(batch, 3, 32, 32))
    activation_totals.append(sum(activation_bytes.values()))
    print(f"batch={batch:4d}  activations={activation_totals[-1]:9,d} B  "
          f"total={fixed_bytes + activation_totals[-1]:9,d} B "
          f"({(fixed_bytes + activation_totals[-1])/1e6:.3f} MB)")

# Activation memory should scale exactly linearly with batch size.
per_example = activation_totals[0]
for batch, total in zip(batch_sizes, activation_totals):
    assert total == per_example * batch, (batch, total, per_example)

fig, ax = plt.subplots()
ax.plot(batch_sizes, [fixed_bytes] * len(batch_sizes), "o--", label="fixed (params+grads+optim)")
ax.plot(batch_sizes, activation_totals, "o-", label="activations (measured)")
ax.set_xlabel("batch size"); ax.set_ylabel("bytes"); ax.set_yscale("log")
ax.set_title("Fixed vs. batch-dependent training memory"); ax.legend()
plt.show()

The fixed cost — 16 bytes per parameter, exactly, whatever the batch size —
never moves in the plot above; only the activation curve grows, and it grows
*exactly* linearly (asserted, not eyeballed): the batch-1 activation byte
count multiplied by 128 matches the measured batch-128 total exactly, because
every one of those extra images produces its own independent copy of every
intermediate feature map. A model that just barely fits its parameters,
gradients and optimiser state on a device can still run out of memory purely
by choosing too large a batch — the fixed cost sets a floor, the batch size
sets how much further above that floor a single step reaches.

## Mixed Precision

`fp32` uses 8 exponent bits and 23 mantissa bits, giving it both a huge
dynamic range and fine precision. `fp16` trims this to 5 exponent bits and 10
mantissa bits: an instant halving of every byte count in the accounting above
(2 bytes vs 4), and faster arithmetic on hardware built for it — at the cost
of a much narrower range, roughly $6\times10^{-5}$ to $65504$ before hitting
`fp16`'s smallest normal or largest representable value. Late in training,
many gradients are far smaller than that: they **underflow to exactly
zero**, and the corresponding parameter simply stops learning. **Loss
scaling** is the fix — multiply the loss by a constant $S$ before the
backward pass (every gradient scales by the same $S$, by linearity of
differentiation), then divide the resulting `fp16` gradients by $S$ back in
`fp32` before the optimiser step. This shifts small gradients into `fp16`'s
representable range without changing what the optimiser ultimately applies.

Separately, some computations must stay in a wider format regardless of loss
scaling: anything that **accumulates many values into one** (a softmax
denominator, a batch-normalisation running mean, the running sum inside the
optimiser's own weight update) compounds rounding error with every addition,
and `fp16`'s 10-bit mantissa cannot absorb that the way `fp32`'s 23 bits can.
This is why mixed-precision training keeps a `fp32` master copy of the
weights and does reductions in `fp32`, and only computes the matrix
multiplications themselves in `fp16`.

In [ ]:
rng = np.random.default_rng(SEED)

# A realistic late-training gradient: mostly very small values.
grad_fp32 = rng.normal(scale=3e-8, size=100_000).astype(np.float32)
grad_fp16 = grad_fp32.astype(np.float16)
frac_underflow = np.mean((grad_fp32 != 0) & (grad_fp16 == 0))
print(f"fraction of nonzero fp32 gradients that underflow to exact zero in fp16: {frac_underflow:.3f}")

SCALE = 65536.0  # a typical starting loss-scale factor
grad_scaled_fp16 = (grad_fp32 * SCALE).astype(np.float16)
frac_underflow_scaled = np.mean((grad_fp32 != 0) & (grad_scaled_fp16 == 0))
grad_recovered = grad_scaled_fp16.astype(np.float32) / SCALE
rel_err = np.abs(grad_recovered - grad_fp32) / (np.abs(grad_fp32) + 1e-12)
print(f"fraction underflowing after loss-scaling by {SCALE:.0f}: {frac_underflow_scaled:.3f}")
print(f"recovered-gradient median relative error vs. the true fp32 value: {np.median(rel_err):.2e}")
assert frac_underflow > 0.5 and frac_underflow_scaled == 0.0

In [ ]:
# Accumulation: summing many small fp16 values loses precision that fp32 keeps.
values = rng.normal(scale=1.0, size=20_000).astype(np.float32)
true_sum = values.astype(np.float64).sum()
sum_fp32 = np.float32(0.0)
sum_fp16 = np.float16(0.0)
for v in values:
    sum_fp32 += v
    sum_fp16 += np.float16(v)
print(f"true (fp64) sum: {true_sum:.4f}")
print(f"fp32 running sum: {float(sum_fp32):.4f}  abs error: {abs(float(sum_fp32) - true_sum):.4f}")
print(f"fp16 running sum: {float(sum_fp16):.4f}  abs error: {abs(float(sum_fp16) - true_sum):.4f}")
assert abs(float(sum_fp32) - true_sum) < abs(float(sum_fp16) - true_sum)

Without loss scaling, 68% of these (deliberately tiny, but realistic for a
nearly-converged parameter) gradients underflow to exactly zero in `fp16` —
the corresponding weights would simply stop updating. Scaling the loss by
65,536 before casting recovers every one of them (0% underflow) at a median
relative error of under 0.02% once unscaled back to `fp32` — the constant is
chosen large enough to lift small gradients into `fp16`'s representable
range without being so large that legitimate large gradients overflow.
Separately, summing the same 20,000 values one at a time in `fp16` drifts
measurably from the true sum where the `fp32` running total does not — the
reason optimiser updates and normalisation statistics are kept in `fp32`
even inside an otherwise `fp16`/`bf16` training step.

## Quantisation

Quantisation maps a tensor's continuous range onto a small set of integers.
For signed int8, the representable integers run from $q_{\min}=-128$ to
$q_{\max}=127$. Given a tensor with observed range $[x_{\min}, x_{\max}]$,
the **scale** is the number of real units one integer step covers,
$$s = \frac{x_{\max}-x_{\min}}{q_{\max}-q_{\min}},$$
and the **zero-point** $z$ is the integer that the real value $0$ maps to
(needed because the real range is rarely symmetric around zero),
$$z = \mathrm{round}\left(q_{\min} - \frac{x_{\min}}{s}\right).$$
Quantising rounds a value onto the integer grid,
$q = \mathrm{clip}(\mathrm{round}(x/s + z),\, q_{\min}, q_{\max})$;
dequantising reverses the affine map, $\hat x = s\,(q - z)$. The error
$|\hat x - x|$ is at most half a step, $s/2$, for any value that did not
clip — the finer the step, the smaller the worst-case error. The step $s$ is
set by the tensor's *entire* observed range, which is exactly where
quantisation loses the most accuracy: a single outlier value forces $s$ to
grow to cover it, coarsening the step size — and therefore the error — for
every other, typical value in the same tensor.

In [ ]:
def quantize_int8(x, qmin=-128, qmax=127):
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / (qmax - qmin)
    zero_point = np.clip(np.round(qmin - x_min / scale), qmin, qmax)
    q = np.clip(np.round(x / scale + zero_point), qmin, qmax).astype(np.int8)
    return q, scale, zero_point


def dequantize_int8(q, scale, zero_point):
    return scale * (q.astype(np.float32) - zero_point)


# Quantise this notebook's own TinyCNN conv1 weights -- a real, trained-scale tensor.
w = model.conv1.weight.detach().numpy().ravel()
q, scale, zp = quantize_int8(w)
w_hat = dequantize_int8(q, scale, zp)
err = np.abs(w_hat - w)
print(f"conv1 weights: {w.size} values, scale={scale:.6f}, zero_point={zp:.1f}")
print(f"  max abs error={err.max():.6f}  mean abs error={err.mean():.6f} "
      f"({err.mean()/w.std():.4f} of the weight std)")
print(f"  memory: fp32={w.nbytes:,} B, int8={q.nbytes:,} B, reduction={w.nbytes/q.nbytes:.1f}x")
assert w.nbytes / q.nbytes == 4.0

In [ ]:
# A single outlier forces the scale -- and therefore the error on every other
# value -- to grow, even though only one value in the tensor changed.
w_outlier = w.copy()
w_outlier[0] = w.std() * 100  # one weight 100 standard deviations out
q2, scale2, zp2 = quantize_int8(w_outlier)
w_hat2 = dequantize_int8(q2, scale2, zp2)
err2 = np.abs(w_hat2 - w_outlier)
print(f"scale without outlier: {scale:.6f}   scale with one outlier: {scale2:.6f} "
      f"({scale2/scale:.1f}x larger)")
print(f"bulk (non-outlier) mean abs error without: {err.mean():.6f}   with: {err2[1:].mean():.6f} "
      f"({err2[1:].mean()/err.mean():.1f}x larger)")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
axes[0].hist(err, bins=40); axes[0].set_title("quantisation error, no outlier")
axes[1].hist(err2[1:], bins=40); axes[1].set_title("quantisation error, with one outlier")
for ax in axes:
    ax.set_xlabel("|dequantised - original|")
plt.tight_layout(); plt.show()
assert scale2 > 5 * scale

Quantising the network's own trained weights to int8 costs almost nothing —
a mean error under 1% of the weight distribution's own spread — at an exact
4x memory reduction (4 bytes down to 1). Injecting a single outlier value
(100 standard deviations from the rest of the tensor) grows the scale more
than 5x and the *bulk* error by roughly the same factor, even though every
other weight in the tensor is completely unchanged: the outlier alone forced
every integer step to cover more real-valued ground. This is why real
quantisation pipelines clip or handle outlier channels specially rather than
letting one extreme value degrade the resolution available to everything
else.

## Distillation

Training a small model directly on hard labels only tells it which class is
correct — a one-hot target carries no information about how *wrong* the
other classes are relative to each other. A trained teacher's own output
distribution does: an image the teacher scores 90% cat, 8% dog, 2%
everything-else says something about *visual similarity* that "the label is
cat" alone does not. **Knowledge distillation** trains the student against a
blend of the true label and the teacher's softened distribution,
$$\mathcal L = \alpha\,\mathrm{CE}(y, \sigma(z_s)) + (1-\alpha)\,T^2\,\mathrm{KL}\!\big(\sigma(z_t/T)\,\|\,\sigma(z_s/T)\big),$$
where $z_s, z_t$ are the student's and teacher's logits and $\sigma$ is the
softmax. The **temperature** $T>1$ divides both sets of logits before the
softmax, flattening the distribution so the "wrong-class" probabilities the
hard label discards are large enough to carry a gradient signal; the $T^2$
factor rescales that term's gradient magnitude back to the same order as the
hard-label term, because temperature-scaling the softmax also scales its
gradient by $1/T$. $\alpha$ trades the two terms off directly.

In [ ]:
class Teacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        return self.fc(x.flatten(1))


class Student(nn.Module):
    """Deliberately much smaller than the teacher: one conv layer, fewer channels."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.pool = nn.MaxPool2d(4)
        self.fc = nn.Linear(8 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        return self.fc(x.flatten(1))


def accuracy(net, X, y):
    with torch.no_grad():
        return (net(X).argmax(1) == y).float().mean().item()


def train_plain(net, epochs, lr=1e-3, batch=64):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    n = len(X_train_t)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            F.cross_entropy(net(X_train_t[idx]), y_train_t[idx]).backward()
            opt.step()
    return net


def train_distill(net, teacher, epochs, lr=1e-3, batch=64, alpha=0.3, T=4.0):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    n = len(X_train_t)
    teacher.eval()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            student_logits = net(X_train_t[idx])
            with torch.no_grad():
                teacher_logits = teacher(X_train_t[idx])
            hard = F.cross_entropy(student_logits, y_train_t[idx])
            soft = F.kl_div(F.log_softmax(student_logits / T, 1),
                             F.softmax(teacher_logits / T, 1), reduction="batchmean") * T ** 2
            (alpha * hard + (1 - alpha) * soft).backward()
            opt.step()
    return net


n_teacher_params = sum(p.numel() for p in Teacher().parameters())
n_student_params = sum(p.numel() for p in Student().parameters())
print(f"teacher: {n_teacher_params:,} params, student: {n_student_params:,} params "
      f"({n_teacher_params/n_student_params:.1f}x fewer)")

The student is trained two ways, five times each with different seeds: once
on hard labels alone, once with the distillation loss above against a single
shared teacher. Test accuracy is one thing to compare — but it is a noisy,
indirect consequence of the loss, especially at this toy scale. What the
distillation loss *directly* optimises for is agreement with the teacher's
output distribution, so that is measured explicitly too: top-1 agreement
with the teacher's own predictions, and KL divergence from the teacher's
full probability distribution.

In [ ]:
teacher = Teacher()
train_plain(teacher, epochs=15)
teacher_acc = accuracy(teacher, X_test_t, y_test_t)
with torch.no_grad():
    teacher_logits_test = teacher(X_test_t)
    teacher_preds = teacher_logits_test.argmax(1)
    teacher_probs = F.softmax(teacher_logits_test, dim=1)
print(f"teacher test accuracy: {teacher_acc:.3f}")

ALPHA, T, EPOCHS = 0.3, 4.0, 15
results = {"acc_plain": [], "acc_distill": [], "agree_plain": [], "agree_distill": [],
           "kl_plain": [], "kl_distill": []}
for seed in range(5):
    torch.manual_seed(seed)
    student_plain = Student()
    train_plain(student_plain, epochs=EPOCHS)
    torch.manual_seed(seed)
    student_distill = Student()
    train_distill(student_distill, teacher, epochs=EPOCHS, alpha=ALPHA, T=T)

    with torch.no_grad():
        sp_logits, sd_logits = student_plain(X_test_t), student_distill(X_test_t)
        results["agree_plain"].append((sp_logits.argmax(1) == teacher_preds).float().mean().item())
        results["agree_distill"].append((sd_logits.argmax(1) == teacher_preds).float().mean().item())
        results["kl_plain"].append(F.kl_div(F.log_softmax(sp_logits, 1), teacher_probs, reduction="batchmean").item())
        results["kl_distill"].append(F.kl_div(F.log_softmax(sd_logits, 1), teacher_probs, reduction="batchmean").item())
    results["acc_plain"].append(accuracy(student_plain, X_test_t, y_test_t))
    results["acc_distill"].append(accuracy(student_distill, X_test_t, y_test_t))

summary = {k: float(np.mean(v)) for k, v in results.items()}
for k, v in summary.items():
    print(f"{k}: {v:.3f}")
assert summary["agree_distill"] > summary["agree_plain"]
assert summary["kl_distill"] < summary["kl_plain"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
labels = ["plain", "distilled"]
axes[0].bar(labels, [summary["agree_plain"], summary["agree_distill"]], color=["#888", "#4477aa"])
axes[0].set_title("agreement with teacher's\ntop-1 prediction"); axes[0].set_ylim(0, 1)
axes[1].bar(labels, [summary["kl_plain"], summary["kl_distill"]], color=["#888", "#4477aa"])
axes[1].set_title("KL divergence from\nteacher's distribution")
plt.tight_layout(); plt.show()

Averaged over 5 seeds, the distilled student agrees with the teacher's top-1
prediction noticeably more often than the plainly-trained student, and its
output distribution sits measurably closer to the teacher's (lower KL) —
both true in every one of the 5 seeds individually, exactly what the
distillation loss was built to optimise. Mean test accuracy, by contrast,
does **not** show a clean win for distillation here (it is a small, noisy toy
problem with only 2,000 training images and 15 epochs) — worth stating
plainly rather than tuning hyperparameters until the toy experiment happens
to agree with the intended story. Distillation reliably transfers the
teacher's *behaviour*; whether that improves accuracy on a specific small,
noisy problem is a separate, empirical question.

## Parallelism Strategies

**Data parallelism** replicates the *entire* model — every one of the bytes
accounted for in "Compute and Memory Accounting" above — on every device,
and splits only the *batch*: each device computes a forward/backward pass on
a different shard of the same batch, then all devices average their
gradients (an all-reduce) before an identical optimiser step runs on every
replica. It scales training **throughput** — more devices process more
examples per second — but does nothing for a model that does not fit on one
device in the first place, because every device still needs the full
$16P$ bytes above (plus its own share of activations). **Model parallelism**
solves exactly that: it splits the *model itself* across devices — different
layers, or even different slices of the same layer's weight matrix — so each
device holds only a fraction of the parameters, gradients and optimiser
state. Its cost is communication: activations, not just gradients, must now
cross the boundary between devices on every forward and backward pass, at
whichever layer the model was split. The two are not mutually exclusive —
real large-scale training typically combines both — but the reason to reach
for model parallelism *specifically* is memory, not speed.

In [ ]:
GPU_MEMORY_BYTES = 24 * 1024 ** 3  # a single 24 GB GPU, a common workstation card


def fixed_training_bytes(n_params, bytes_per_param=4, optimizer_multiplier=2):
    return n_params * bytes_per_param * (2 + optimizer_multiplier)  # params + grads + optim state


scenarios = [
    ("this notebook's TinyCNN", n_params),
    ("a 350M-parameter model", 350_000_000),
    ("a 7B-parameter model", 7_000_000_000),
]

names, gb_needed, gpus_needed = [], [], []
for name, p in scenarios:
    total_bytes = fixed_training_bytes(p)
    n_gpus = int(np.ceil(total_bytes / GPU_MEMORY_BYTES))
    names.append(name); gb_needed.append(total_bytes / 1024 ** 3); gpus_needed.append(n_gpus)
    print(f"{name}: {p:,} params -> {total_bytes/1024**3:.2f} GB fixed memory -> "
          f"fits one {GPU_MEMORY_BYTES/1024**3:.0f} GB GPU: {total_bytes <= GPU_MEMORY_BYTES}, "
          f"min GPUs to shard the model itself: {n_gpus}")

fig, ax = plt.subplots()
ax.bar(names, gb_needed, color=["#4477aa", "#4477aa", "#cc6677"])
ax.axhline(GPU_MEMORY_BYTES / 1024 ** 3, color="k", linestyle="--", label="single 24 GB GPU")
ax.set_ylabel("GB, fixed training memory"); ax.set_yscale("log"); ax.legend()
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()
assert gpus_needed[-1] > 1 and gpus_needed[0] == 1

This notebook's own TinyCNN and even a 350M-parameter model comfortably fit
their fixed training memory on a single 24 GB GPU; a 7B-parameter model's
parameters, gradients and Adam state alone need over 100 GB — more than four
such GPUs' worth — *before a single activation is stored or a single training
example is processed*. Data parallelism cannot help here: adding more
replicas of a model that does not fit on one device does not make it fit.
Model parallelism is the only one of the two that changes whether training is
possible at all; it is only after the model itself fits that data parallelism
becomes the right tool for making training *faster*.

## Key Takeaways

- **A training step's fixed memory is $16P$ bytes for $P$ `fp32` Adam
  parameters** (params + grads + 2x optimiser state), confirmed exactly on
  this notebook's TinyCNN; **activation memory is the only part that depends
  on batch size**, and forward hooks confirmed it scales exactly linearly.
- **`fp16` silently underflows small gradients to zero** — 68% of a
  realistic late-training gradient tensor vanished in this notebook's
  demonstration — and **loss scaling recovers all of them** at under 0.02%
  relative error; **accumulation (sums, running statistics) must stay in
  `fp32`** regardless, because narrow-mantissa accumulation drifts from the
  true sum in a way loss scaling does not fix.
- **int8 quantisation's scale and zero-point derive directly from a tensor's
  observed range**, cost this notebook's real conv weights well under 1%
  relative error at an exact 4x memory reduction, and **a single 100-sigma
  outlier grew the scale, and the error on every other value, by 5x or more**
  — quantisation error is a property of the whole tensor, not of each value
  independently.
- **Distillation's loss directly optimises agreement with the teacher's
  output distribution**, and delivered exactly that (higher top-1 agreement,
  lower KL divergence, in every one of 5 seeds) — while mean test accuracy on
  this small, noisy CIFAR-10 subset did not show a clean win, a result worth
  reporting honestly rather than tuned away.
- **Data parallelism replicates the whole model and splits the batch; model
  parallelism splits the model itself.** This notebook's own memory
  arithmetic showed a 7B-parameter model needs over 100 GB of fixed training
  memory — more than four 24 GB GPUs — making model parallelism a memory
  necessity, not merely a speed optimisation, for models at that scale.